# Generate Additional Emotion Story Datasets

This notebook runs the story generator for the additional conflict-avoidance emotions. It works both in Google Colab and from a local Jupyter server opened inside this repo.

It uses the canonical generator at `scripts/generate_datasets.py` and patches `emotion_mechanisms.config` in the notebook before loading the generator, so you do not need to edit source files in the notebook runtime.

Use `hf` on Colab. If you do not have a very large GPU, start with `Qwen/Qwen2.5-7B-Instruct` or `Qwen/Qwen2.5-14B-Instruct` instead of `32B`.


## 1. Repo and Runtime

Run the next three cells first. In Colab they clone the repo into `/content`. In local Jupyter they use your existing checkout and avoid `/content`, which is Colab-specific.


In [2]:
import os
import subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/daspushpita/emotion-mechanisms-llm.git"

if IN_COLAB:
    REPO_DIR = Path("/content/emotion-mechanisms-llm")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    current = Path.cwd().resolve()
    REPO_DIR = current.parent if current.name == "notebooks" else current

os.chdir(REPO_DIR)
print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_DIR={REPO_DIR}")


IN_COLAB=True
REPO_DIR=/content/emotion-mechanisms-llm


In [3]:
import sys
import subprocess

if IN_COLAB:
    subprocess.run([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "transformers",
        "accelerate",
        "huggingface_hub",
        "sentencepiece",
    ], check=True)
else:
    print("Skipping Colab package install. Use your local environment packages.")


In [4]:
import shutil
import subprocess

if IN_COLAB and shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("Skipping nvidia-smi outside Colab or when unavailable.")


Skipping nvidia-smi outside Colab or when unavailable.


## 2. Configure the Run

Edit the values in the next cell before you launch generation.

- `HF_MODEL_ID`: pick the Hugging Face model to use on Colab.
- `STORY_GENERATION_MODE`: use `"emotional_stories"` for the additional emotions; switch to `"neutral_stories"` if you want to generate the matching neutral file.
- `ADDITIONAL_EMOTIONS`: the emotion labels to generate.
- `LIMIT_TOPICS`: set this to a small number for a smoke test.
- `SAMPLES_PER_TOPIC_EMOTION`, `NEUTRAL_PER_TOPIC`, `STORIES_PER_BATCH`, and `MAX_NEW_TOKENS`: optional overrides for a smaller or faster run.
- `MOUNT_DRIVE`: mount Google Drive so you can copy the final JSONL out of ephemeral Colab storage.


In [5]:
HF_MODEL_ID = "Qwen/Qwen2.5-32B-Instruct"
TOPICS_FILE = "datasets/raw/topics_v1.txt"
DATA_ROOT = Path('/content/drive/MyDrive/emotion-mechanisms-llm') if IN_COLAB else REPO_DIR

STORY_GENERATION_MODE = "emotional_stories"  # "emotional_stories" or "neutral_stories"
ADDITIONAL_EMOTIONS = [
    "deferential",
    "conflict_avoidant",
    "socially_anxious",
    "ashamed",
    "approval_seeking",
    "people_pleasing",
    "validation_seeking",
    "submissive",
    "obsequious"
]

EMOTIONAL_OUTPUT_FILE = "additional_emotional_stories_qwen32B_v2.jsonl"
NEUTRAL_OUTPUT_FILE = "neutral_stories_qwen32B_v2.jsonl"

LIMIT_TOPICS = None
SAMPLES_PER_TOPIC_EMOTION = 12
NEUTRAL_PER_TOPIC = 3
STORIES_PER_BATCH = 5
MAX_NEW_TOKENS = 1600
GENERATION_PARSE_RETRIES = None

MOUNT_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/emotion-mechanisms-llm/datasets/processed"


In [6]:
from pathlib import Path

if MOUNT_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    DRIVE_OUTPUT_DIR = Path(DRIVE_OUTPUT_DIR)
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Drive output dir: {DRIVE_OUTPUT_DIR}")
elif MOUNT_DRIVE and not IN_COLAB:
    DRIVE_OUTPUT_DIR = None
    print("Skipping Google Drive mount outside Colab.")
else:
    DRIVE_OUTPUT_DIR = None
    print("Skipping Google Drive mount.")


Mounted at /content/drive
Drive output dir: /content/drive/MyDrive/emotion-mechanisms-llm/datasets/processed


## 3. Optional Hugging Face Login

Run the next cell only if the model download needs authentication or you want authenticated rate limits.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


## 4. Run Dataset Generation

The next two cells patch the runtime config for this notebook session and then run `scripts/generate_datasets.py` directly.


In [ ]:
import importlib.util
import os
import sys
from pathlib import Path

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import emotion_mechanisms.config as cfg
import emotion_mechanisms.model_loader as model_loader

topics_path = DATA_ROOT / TOPICS_FILE

if LIMIT_TOPICS is not None:
    subset_path = DATA_ROOT / "datasets/raw/topics_colab_subset.txt"
    topics = [line.strip() for line in topics_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    subset_path.write_text("\n".join(topics[:LIMIT_TOPICS]) + "\n", encoding="utf-8")
    topics_path = subset_path
    print(f"Using a subset of {LIMIT_TOPICS} topics: {topics_path}")
else:
    print(f"Using full topics file: {topics_path}")

processed_dir = DATA_ROOT / "datasets/processed"
processed_dir.mkdir(parents=True, exist_ok=True)

model_loader.HF_MODEL = HF_MODEL_ID
cfg.GENERATION_BACKEND = "hf"
cfg.TOPICS_PATH = topics_path
cfg.PROCESSED_DIR = processed_dir
cfg.STORY_GENERATION_MODE = STORY_GENERATION_MODE
cfg.EMOTIONS = ADDITIONAL_EMOTIONS
cfg.CONFLICT_AVOIDANCE_EMOTIONS = ADDITIONAL_EMOTIONS
cfg.ALL_EMOTIONS = ADDITIONAL_EMOTIONS
cfg.EMOTIONAL_STORIES_DATASET = processed_dir / EMOTIONAL_OUTPUT_FILE
cfg.NEUTRAL_STORIES_DATASET = processed_dir / NEUTRAL_OUTPUT_FILE

if SAMPLES_PER_TOPIC_EMOTION is not None:
    cfg.N_SAMPLES_PER_TOPIC_EMOTION = SAMPLES_PER_TOPIC_EMOTION
if NEUTRAL_PER_TOPIC is not None:
    cfg.N_NEUTRAL_PER_TOPIC = NEUTRAL_PER_TOPIC
if STORIES_PER_BATCH is not None:
    cfg.N_STORIES_PER_BATCH = STORIES_PER_BATCH
if MAX_NEW_TOKENS is not None:
    cfg.MAX_NEW_TOKENS = MAX_NEW_TOKENS
if GENERATION_PARSE_RETRIES is not None:
    cfg.GENERATION_PARSE_RETRIES = GENERATION_PARSE_RETRIES

output_path = (
    cfg.EMOTIONAL_STORIES_DATASET
    if cfg.STORY_GENERATION_MODE == "emotional_stories"
    else cfg.NEUTRAL_STORIES_DATASET
)

print(f"HF_MODEL_ID={model_loader.HF_MODEL}")
print(f"STORY_GENERATION_MODE={cfg.STORY_GENERATION_MODE}")
print(f"TOPICS_PATH={cfg.TOPICS_PATH}")
print(f"EMOTIONS={cfg.EMOTIONS}")
print(f"OUTPUT_PATH={output_path}")
print(f"SAMPLES_PER_TOPIC_EMOTION={cfg.N_SAMPLES_PER_TOPIC_EMOTION}")
print(f"N_NEUTRAL_PER_TOPIC={cfg.N_NEUTRAL_PER_TOPIC}")
print(f"N_STORIES_PER_BATCH={cfg.N_STORIES_PER_BATCH}")
print(f"MAX_NEW_TOKENS={cfg.MAX_NEW_TOKENS}")


In [ ]:
generator_path = REPO_DIR / "scripts/generate_datasets.py"
spec = importlib.util.spec_from_file_location("generate_dataset", generator_path)
generator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(generator)
generator.main()


## 5. Inspect and Persist the Output

This shows the generated file selected by `STORY_GENERATION_MODE` and optionally copies it into Google Drive.


In [ ]:
import shutil

output_path = (
    cfg.EMOTIONAL_STORIES_DATASET
    if cfg.STORY_GENERATION_MODE == "emotional_stories"
    else cfg.NEUTRAL_STORIES_DATASET
)

print(f"Local output: {output_path}")
print(f"Exists: {output_path.exists()}")

if output_path.exists():
    !wc -l {output_path}
    !head -n 2 {output_path}

if DRIVE_OUTPUT_DIR is not None and output_path.exists():
    target_path = DRIVE_OUTPUT_DIR / output_path.name
    shutil.copy2(output_path, target_path)
    print(f"Copied to: {target_path}")
